# Demostración: Descomposición Intrínseca Multiescala (Retinex)

Este notebook importa la lógica de negocio pura de `src/` para ejecutar visualizaciones y comparaciones interactivas entre las transformadas **Starlet** y **MMT (Multiscale Median Transform)** bajo las reglas avanzadas de **coherencia inter-escala** y **coherencia de color**.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import os
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

# Añadir la raíz del proyecto al path para importar src
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from src.data.mit import MITIntrinsicDataset
from src.decompose.multiscale import multiscale_decomposition
from src.metrics.lmse import local_error

### 1. Cargar Objeto de Prueba
Cargaremos la imagen del objeto `box` del dataset de MIT Intrinsic Images.

In [ ]:
dataset = MITIntrinsicDataset()
data = dataset.load_object('box')

diffuse = data['diffuse']       # Imagen de entrada RGB
true_refl = data['reflectance'] # Reflectancia de referencia (GT)
true_shading = data['shading']   # Shading de referencia (GT)
mask = data['mask']             # Máscara binaria del objeto

### 2. Ejecutar Descomposiciones Multiescala
Comparamos la transformada de Starlet (con coherencia de color) frente al MMT completo (con coherencia de color y escala).

In [ ]:
# 1. Starlet con coherencia de color
shading_starlet, refl_starlet = multiscale_decomposition(
    diffuse, mask, levels=3, threshold_factor=2.0, transform_type="starlet",
    color_coherence=True, color_threshold=0.05, color_beta=0.5
)

# 2. MMT Completo (mediana + escala + color)
shading_mmt, refl_mmt = multiscale_decomposition(
    diffuse, mask, levels=3, threshold_factor=2.0, transform_type="mmt",
    scale_coherence=True, color_coherence=True, color_threshold=0.05, color_beta=0.5
)

### 3. Visualización y Comparación
Graficamos la imagen original, la reflectancia y el shading estimado para ambos métodos frente al Ground Truth.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(15, 12))

# Fila 0: Entrada y Ground Truth
axes[0, 0].imshow(diffuse)
axes[0, 0].set_title("Imagen de Entrada (box)")
axes[0, 0].axis('off')

axes[0, 1].imshow(true_refl)
axes[0, 1].set_title("Reflectancia Referencia (GT)")
axes[0, 1].axis('off')

axes[0, 2].imshow(true_shading, cmap='gray')
axes[0, 2].set_title("Shading Referencia (GT)")
axes[0, 2].axis('off')

# Fila 1: Estimaciones con Starlet + Color
axes[1, 0].imshow(diffuse)
axes[1, 0].set_title("Imagen de Entrada")
axes[1, 0].axis('off')

axes[1, 1].imshow(np.clip(refl_starlet, 0.0, 1.0))
axes[1, 1].set_title("Reflectancia Starlet + Color")
axes[1, 1].axis('off')

axes[1, 2].imshow(np.clip(shading_starlet, 0.0, 1.0), cmap='gray')
axes[1, 2].set_title("Shading Starlet + Color")
axes[1, 2].axis('off')

# Fila 2: Estimaciones con MMT Completo
axes[2, 0].imshow(diffuse)
axes[2, 0].set_title("Imagen de Entrada")
axes[2, 0].axis('off')

axes[2, 1].imshow(np.clip(refl_mmt, 0.0, 1.0))
axes[2, 1].set_title("Reflectancia MMT Completo")
axes[2, 1].axis('off')

axes[2, 2].imshow(np.clip(shading_mmt, 0.0, 1.0), cmap='gray')
axes[2, 2].set_title("Shading MMT Completo")
axes[2, 2].axis('off')

plt.tight_layout()
plt.show()

### 4. Métricas Cuantitativas
Calculamos el error LMSE para evaluar la precisión física de la descomposición.

In [ ]:
# Convertimos a escala de grises para el cálculo de LMSE
true_refl_gray = np.mean(true_refl, axis=2)
true_shading_gray = np.mean(true_shading, axis=2) if true_shading.ndim == 3 else true_shading

refl_starlet_gray = np.mean(refl_starlet, axis=2)
shading_starlet_gray = np.mean(shading_starlet, axis=2) if shading_starlet.ndim == 3 else shading_starlet

refl_mmt_gray = np.mean(refl_mmt, axis=2)
shading_mmt_gray = np.mean(shading_mmt, axis=2) if shading_mmt.ndim == 3 else shading_mmt

# LMSE
lmse_r_starlet = local_error(true_refl_gray, refl_starlet_gray, mask)
lmse_s_starlet = local_error(true_shading_gray, shading_starlet_gray, mask)
lmse_comb_starlet = 0.5 * lmse_r_starlet + 0.5 * lmse_s_starlet

lmse_r_mmt = local_error(true_refl_gray, refl_mmt_gray, mask)
lmse_s_mmt = local_error(true_shading_gray, shading_mmt_gray, mask)
lmse_comb_mmt = 0.5 * lmse_r_mmt + 0.5 * lmse_s_mmt

print("=================== COMPARATIVA LMSE ===================")
print(f"Starlet + Color Combined LMSE: {lmse_comb_starlet:.4f} (Refl: {lmse_r_starlet:.4f}, Shad: {lmse_s_starlet:.4f})")
print(f"MMT Completo Combined LMSE:    {lmse_comb_mmt:.4f} (Refl: {lmse_r_mmt:.4f}, Shad: {lmse_s_mmt:.4f})")
print("========================================================")